# Spark Silver Processing

This notebook is designed to be run **headlessly** by Airflow via `jupyter nbconvert --execute`.
It reads Bronze data (schedule, matchsheet, lineup) from MinIO and builds Silver Iceberg tables.

**Do not add extraction or upload cells here.** The DAG handles that separately.

In [ ]:
# Initialize the Spark session with AQE optimizations
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# The Spark session uses config from spark-defaults.conf (Iceberg + MinIO)
spark = (SparkSession.builder
    .appName("BrasileiraoSilverProcessing")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .getOrCreate())

# Reduce log noise during headless execution
spark.sparkContext.setLogLevel("WARN")
print("Spark session created.")

In [ ]:
# Configuration: Bronze data location in MinIO
RAW_BUCKET = "datalake-raw"
SEASON = 2024
BRONZE_URI = f"s3a://{RAW_BUCKET}/espn/brasileirao/{SEASON}"

# Read the three Bronze JSONs from MinIO
df_schedule = spark.read.option("multiline", "true").json(f"{BRONZE_URI}/schedule.json")
df_matchsheet = spark.read.option("multiline", "true").json(f"{BRONZE_URI}/matchsheet.json")
df_lineup = spark.read.option("multiline", "true").json(f"{BRONZE_URI}/lineup.json")

print(f"Schedule:   {df_schedule.count()} rows")
print(f"Matchsheet: {df_matchsheet.count()} rows")
print(f"Lineup:     {df_lineup.count()} rows")

In [ ]:
# --- Step 1: Derive scores from lineup (sum total_goals per game + team) ---
df_scores = (
    df_lineup
    # Cast total_goals to integer (may be null or string)
    .withColumn("total_goals", F.coalesce(F.col("total_goals").cast("int"), F.lit(0)))
    # Aggregate: sum goals per game + is_home
    .groupBy("game", "is_home")
    .agg(F.sum("total_goals").alias("team_goals"))
)

# Pivot to get home_score and away_score columns per game
df_home_scores = df_scores.filter(F.col("is_home") == True).select(
    F.col("game"), F.col("team_goals").alias("home_score")
)
df_away_scores = df_scores.filter(F.col("is_home") == False).select(
    F.col("game"), F.col("team_goals").alias("away_score")
)

# --- Step 2: Pivot matchsheet to get home/away stats side by side ---
# Stat columns from matchsheet (exclude metadata)
meta_cols = {"league", "season", "game", "team", "is_home", "venue", "attendance", "capacity"}
stat_cols = [c for c in df_matchsheet.columns if c not in meta_cols]

# Home team stats (prefix with home_)
df_home_stats = df_matchsheet.filter(F.col("is_home") == True)
for col_name in stat_cols:
    df_home_stats = df_home_stats.withColumnRenamed(col_name, f"home_{col_name}")
df_home_stats = df_home_stats.select(
    "game", "venue", "attendance", "capacity",
    *[f"home_{c}" for c in stat_cols]
)

# Away team stats (prefix with away_)
df_away_stats = df_matchsheet.filter(F.col("is_home") == False)
for col_name in stat_cols:
    df_away_stats = df_away_stats.withColumnRenamed(col_name, f"away_{col_name}")
df_away_stats = df_away_stats.select(
    "game",
    *[f"away_{c}" for c in stat_cols]
)

# --- Step 3: Join everything into one enriched match table ---
df_silver = (
    df_schedule
    # Join home scores
    .join(df_home_scores, on="game", how="left")
    # Join away scores
    .join(df_away_scores, on="game", how="left")
    # Join home team stats + venue/attendance
    .join(df_home_stats, on="game", how="left")
    # Join away team stats
    .join(df_away_stats, on="game", how="left")
)

# --- Step 4: Add computed analytics columns ---
df_silver = (
    df_silver
    .withColumn("home_score", F.col("home_score").cast("int"))
    .withColumn("away_score", F.col("away_score").cast("int"))
    .withColumn("total_goals", F.col("home_score") + F.col("away_score"))
    .withColumn("goal_diff", F.abs(F.col("home_score") - F.col("away_score")))
    .withColumn("is_draw", F.col("home_score") == F.col("away_score"))
    .withColumn("winner", F.when(
        F.col("home_score") > F.col("away_score"), F.col("home_team")
    ).when(
        F.col("away_score") > F.col("home_score"), F.col("away_team")
    ).otherwise(F.lit("Draw")))
    .withColumn("processed_at", F.current_timestamp())
)

print(f"Enriched Silver table: {df_silver.count()} rows, {len(df_silver.columns)} columns")

In [ ]:
# Build player match stats Silver table from lineup data
numeric_cols = [
    "total_goals", "goal_assists", "shots_on_target", "total_shots",
    "fouls_committed", "fouls_suffered", "yellow_cards", "red_cards",
    "own_goals", "offsides", "appearances", "saves", "shots_faced",
    "goals_conceded"
]

# Cast numeric columns, filling nulls with 0
df_player_stats = df_lineup
for col_name in numeric_cols:
    if col_name in df_player_stats.columns:
        df_player_stats = df_player_stats.withColumn(
            col_name,
            F.coalesce(F.col(col_name).cast("int"), F.lit(0))
        )

# Add processing timestamp
df_player_stats = df_player_stats.withColumn("processed_at", F.current_timestamp())

print(f"Player Match Stats: {df_player_stats.count()} rows")

In [ ]:
# Write both Silver Iceberg tables
spark.sql("CREATE NAMESPACE IF NOT EXISTS lake.analytics")

# Write enriched match statistics
match_table = "lake.analytics.match_statistics"
df_silver.writeTo(match_table).createOrReplace()
print(f"Wrote {df_silver.count()} rows to {match_table}")

# Write player match stats
player_table = "lake.analytics.player_match_stats"
df_player_stats.writeTo(player_table).createOrReplace()
print(f"Wrote {df_player_stats.count()} rows to {player_table}")

In [ ]:
# Quick verification queries
match_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {match_table}").collect()[0]["cnt"]
player_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {player_table}").collect()[0]["cnt"]
print(f"Verification: {match_table} has {match_count} rows")
print(f"Verification: {player_table} has {player_count} rows")

# Show top 5 highest-scoring matches
print("\nTop 5 matches by total goals:")
spark.sql(f"""
    SELECT game_id, home_team, away_team, home_score, away_score, total_goals, winner
    FROM {match_table}
    ORDER BY total_goals DESC
    LIMIT 5
""").show(truncate=False)

In [ ]:
# Stop the Spark session to free all memory
spark.stop()
print("Spark session stopped. Memory released.")